## Changelog
- parent: 20260507_192717_00caf657
- change: extend the minimal preprocessing pipeline (median impute + ordinal
    encode) to every column in the train set, not just the top-20 MI features.
- hypothesis: the top-20 baseline regressed to public 17088.61 vs the parent's
    14719.79, so the dropped 59 columns carry signal the GBM was previously using.
    Putting all columns back through the same minimal preprocessing isolates how
    much of the parent's lift came from FE vs. just having the full feature set.

In [ ]:
import sys
import numpy as np
import pandas as pd

from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    data_dir   = Path("/kaggle/input/home-data-for-ml-course")
    output_dir = Path("/kaggle/working")
else:
    def _find_competition_dir(start: Path) -> Path:
        for p in [start, *start.parents]:
            if (p / "config.yaml").exists() and (p / "data").is_dir():
                return p
        raise RuntimeError("competition dir not found (no ancestor has config.yaml + data/)")

    comp_dir   = _find_competition_dir(Path.cwd())
    data_dir   = comp_dir / "data"
    output_dir = Path.cwd()

train_data_raw = pd.read_csv(data_dir / "train.csv")
test_data_raw  = pd.read_csv(data_dir / "test.csv")

In [ ]:
# see eda-TotalSF.ipynb — drop the mega-house outliers (TotalSF > 7000)
_outlier_mask = (
    train_data_raw["TotalBsmtSF"]
    + train_data_raw["1stFlrSF"]
    + train_data_raw["2ndFlrSF"]
) > 7000
train_data = train_data_raw.loc[~_outlier_mask].reset_index(drop=True)

# Drop Id (row identifier, no signal) and SalePrice (target). Everything else
# goes through preprocessing.
DROP = ["Id", "SalePrice"]
X      = train_data.drop(columns=DROP, errors="ignore").copy()
X_test = test_data_raw.drop(columns=DROP, errors="ignore").copy()
y      = np.log1p(train_data["SalePrice"])

# MSSubClass is a nominal int code — cast to string so the encoder treats it
# as a category rather than an ordered number.
X["MSSubClass"]      = X["MSSubClass"].astype(str)
X_test["MSSubClass"] = X_test["MSSubClass"].astype(str)

# Auto-detect num/cat from train; apply the same split to test.
NUMERIC     = X.select_dtypes(include="number").columns.tolist()
CATEGORICAL = X.select_dtypes(exclude="number").columns.tolist()
print(f"{len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical = {len(X.columns)} total")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold

# ColumnTransformer: List of (name, transformer, columns) tuples specifying the transformer objects to be applied to subsets of the data.
# transformer: {‘drop’, ‘passthrough’} or estimator. Estimator must support fit and transform. 
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC),  # Replace missing values using median along each column
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ord",    OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), CATEGORICAL),
])

pipe = Pipeline([
    ("prep",  preprocessor),
    ("model", GradientBoostingRegressor(n_estimators=300, random_state=42)),
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    pipe, X, y,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)
rmse = -scores
print(f"CV RMSE (log-price): {rmse.mean():.4f} ± {rmse.std():.4f}")
print(f"Per-fold:            {np.round(rmse, 4).tolist()}")

pipe.fit(X, y)

In [ ]:
test_pred = np.expm1(pipe.predict(X_test))

sample = pd.read_csv(data_dir / "sample_submission.csv")
submission = sample.copy()
submission["SalePrice"] = test_pred
submission.to_csv(output_dir / "submission.csv", index=False)